In [ ]:
import numpy as np
import pandas as pd
import tempfile
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))
from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

DATA_DIR = PROJECT_DIR / "data"

# --- CONFIGURATION ---
MATCH_ACTUEL_ID = 1      # Le match sur lequel on parie aujourd'hui
MATCH_FUTUR_ID = 15      # Le match futur (J2/J3) dont les cotes sont incertaines
MON_GAP_1 = 0
MON_GAP_2 = 0
HAS_BOOSTER = 1

g1_idx = max(GAP_MIN, min(GAP_MAX, MON_GAP_1)) + GAP_OFFSET
g2_idx = max(GAP_MIN, min(GAP_MAX, MON_GAP_2)) + GAP_OFFSET
MODE = 1 if HAS_BOOSTER else 0   # colonne Q-table : 0=sans booster, 1=keep (safe), 2=use x2


def win_rates(csv_path):
    """WR des 3 issues pour le match du jour (DP déterministe jusqu'à l'horizon).

    use_drift=False -> comparaison reproductible. La rétro-propagation de la DP
    fait remonter l'effet d'un match futur jusqu'à la décision d'aujourd'hui.
    """
    _, _, _, q = run_daily_pipeline(
        csv_path=csv_path,
        match_id_cible=MATCH_ACTUEL_ID,
        mon_gap_1=MON_GAP_1, mon_gap_2=MON_GAP_2,
        has_booster=HAS_BOOSTER,
        use_drift=False,
        horizon_nuit=1,
        nb_scenarios=1,
        save_abaques=False,
    )
    return q[g1_idx, g2_idx, :, MODE]


# --- 1. BASELINE (le monde tel qu'on l'estime aujourd'hui) ---
csv_base = DATA_DIR / "CDM_2026.csv"
print("Calcul baseline...")
wr_base = win_rates(csv_base)

# --- 2. PERTURBATION DU FUTUR (le "crash test") ---
# Le match futur, qu'on pensait équilibré, devient ultra-déséquilibré :
# un favori écrasant apparaît (cotes 1.18 / 10 / 20) et la foule suit massivement.
df = pd.read_csv(csv_base)
idx = MATCH_FUTUR_ID - 1
df.loc[idx, ['cote_1', 'cote_N', 'cote_2']] = [1 / 0.85, 1 / 0.10, 1 / 0.05]
df.loc[idx, ['crowd_1', 'crowd_N', 'crowd_2']] = [0.95, 0.03, 0.02]
csv_pert = Path(tempfile.gettempdir()) / "_mpp_sensibilite.csv"
df.to_csv(csv_pert, index=False)
print("Calcul perturbé...")
wr_pert = win_rates(csv_pert)

# --- 3. AFFICHAGE (L'Effet Papillon) ---
noms = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
print(f"\n--- ANALYSE DE SENSIBILITÉ DU MATCH {MATCH_ACTUEL_ID} ---")
print(f"Perturbation appliquée sur le Match {MATCH_FUTUR_ID} (J2/J3).")
print("-" * 58)
for i in range(3):
    diff = wr_pert[i] - wr_base[i]
    signe = "+" if diff >= 0 else ""
    print(f"Choix {noms[i]:<10} : Baseline {wr_base[i]*100:6.2f}% | "
          f"Perturbé {wr_pert[i]*100:6.2f}% | Écart {signe}{diff*100:.2f}%")

a_base, a_pert = int(np.argmax(wr_base)), int(np.argmax(wr_pert))
print("-" * 58)
if a_base == a_pert:
    print(f"✅ RÉSILIENCE : la décision optimale ({noms[a_base]}) ne change pas malgré le choc futur.")
else:
    print(f"⚠️ EFFET PAPILLON : la décision bascule de '{noms[a_base]}' vers '{noms[a_pert]}'.")
    print("-> Une anomalie en J2/J3 modifie la prise de risque optimale dès aujourd'hui.")